Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install comfy-aimdo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 4.6 MB/s eta 0:00:00


In [ ]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/AI/ComfyUI"
    %cd /content/drive/MyDrive/AI

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Fix: Force reset to official repo to resolve 'comfy_aimdo' errors from prior forks
  !git config --global --add safe.directory $WORKSPACE
  !git remote set-url origin https://github.com/comfyanonymous/ComfyUI
  !git fetch origin
  !git checkout -B master origin/master
  !git reset --hard origin/master

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# Configure ComfyUI to use external models directory
import yaml

extra_model_paths = {
    "comfyui": {
        "base_path": "/content/drive/MyDrive/AI/models",
        "checkpoints": "checkpoints",
        "clip": "clip",
        "clip_vision": "clip_vision",
        "configs": "configs",
        "controlnet": "controlnet",
        "embeddings": "embeddings",
        "loras": "loras",
        "upscale_models": "upscale_models",
        "vae": "vae",
        "llm": "llm",
        "audio_models": "audio_models"
    },
    "diffusion_base_sd": {
        "base_path": "/content/drive/MyDrive/AI/models/diffusion_base",
        "checkpoints": "sd"
    },
    "diffusion_base_sdxl": {
        "base_path": "/content/drive/MyDrive/AI/models/diffusion_base",
        "checkpoints": "sdxl"
    },
    "sd_inpainting": {
        "base_path": "/content/drive/MyDrive/AI",
        "checkpoints": "stable-diffusion-inpainting-model"
    },
    "sdxl_inpainting": {
        "base_path": "/content/drive/MyDrive/AI",
        "checkpoints": "sdxl-inpainting-model"
    }
}

with open(f'{WORKSPACE}/extra_model_paths.yaml', 'w') as f:
    yaml.dump(extra_model_paths, f, default_flow_style=False)


Mounting Google Drive...
/
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/AI
/content/drive/MyDrive/AI/ComfyUI
-= Updating ComfyUI =-
M	.ci/update_windows/update_comfyui_stable.bat
M	.ci/windows_amd_base_files/README_VERY_IMPORTANT.txt
M	.ci/windows_amd_base_files/run_amd_gpu.bat
M	.ci/windows_amd_base_files/run_amd_gpu_disable_smart_memory.bat
M	.ci/windows_nvidia_base_files/README_VERY_IMPORTANT.txt
M	.ci/windows_nvidia_base_files/run_cpu.bat
M	.ci/windows_nvidia_base_files/run_nvidia_gpu.bat
M	comfy/audio_encoders/whisper.py
M	comfy/ldm/ace/model.py
M	comfy/ldm/ace/vae/music_log_mel.py
M	comfy/ldm/ace/vae/music_vocoder.py
M	comfy/ldm/audio/dit.py
M	comfy/ldm/hunyuan3dv2_1/hunyuandit.py
M	comfy/ldm/hunyuan_video/model.py
M	comfy/ldm/kandinsky5/model.py
M	comfy/ldm/wan/model_animate.py
M	comfy/ldm/wan/model_multitalk.py
M	comfy/ldm/wan/vae.py
M	comfy/ldm/wan/vae2_2.py
M	comfy/lora.

Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [ ]:
# Checkpoints
import os

def download(url, path, filename=None):
    if filename:
        dest = os.path.join(path, filename)
    else:
        dest = os.path.join(path, os.path.basename(url))

    if not os.path.exists(dest):
        print(f"Downloading {url} to {dest}...")
        if filename:
             !wget -c {url} -O {dest}
        else:
             !wget -c {url} -P {path}
    else:
        print(f"File {dest} already exists, skipping download.")

# Define paths based on your setup
base_models = "/content/drive/MyDrive/AI/models"
ckpt_path = os.path.join(base_models, "checkpoints")
vae_path = os.path.join(base_models, "vae")
lora_path = os.path.join(base_models, "loras")
controlnet_path = os.path.join(base_models, "controlnet")
style_path = os.path.join(base_models, "style_models")
clip_vision_path = os.path.join(base_models, "clip_vision")
gligen_path = os.path.join(base_models, "gligen")
upscale_path = os.path.join(base_models, "upscale_models")

### SDXL
### I recommend these workflow examples: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/

#download("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors", ckpt_path)
#download("https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors", ckpt_path)

# SDXL ReVision
#download("https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors", clip_vision_path)

# SD1.5
download("https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt", ckpt_path)

# SD2
#download("https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors", ckpt_path)
#download("https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors", ckpt_path)

# Some SD1.5 anime style
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors", ckpt_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1_orangemixs.safetensors", ckpt_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A3_orangemixs.safetensors", ckpt_path)
#download("https://huggingface.co/Linaqruf/anything-v3.0/resolve/main/anything-v3-fp16-pruned.safetensors", ckpt_path)

# Waifu Diffusion 1.5 (anime style SD2.x 768-v)
#download("https://huggingface.co/waifu-diffusion/wd-1-5-beta3/resolve/main/wd-illusion-fp16.safetensors", ckpt_path)


# unCLIP models
#download("https://huggingface.co/comfyanonymous/illuminatiDiffusionV1_v11_unCLIP/resolve/main/illuminatiDiffusionV1_v11-unclip-h-fp16.safetensors", ckpt_path)
#download("https://huggingface.co/comfyanonymous/wd-1.5-beta2_unCLIP/resolve/main/wd-1-5-beta2-aesthetic-unclip-h-fp16.safetensors", ckpt_path)


# VAE
download("https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors", vae_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt", vae_path)
#download("https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt", vae_path)


# Loras
#download("https://civitai.com/api/download/models/10350", lora_path, "theovercomer8sContrastFix_sd21768.safetensors") #theovercomer8sContrastFix SD2.x 768-v
#download("https://civitai.com/api/download/models/10638", lora_path, "theovercomer8sContrastFix_sd15.safetensors") #theovercomer8sContrastFix SD1.x
#download("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors", lora_path) #SDXL offset noise lora


# T2I-Adapter
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_depth_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_seg_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_sketch_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_keypose_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_openpose_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_color_sd14v1.pth", controlnet_path)
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_canny_sd14v1.pth", controlnet_path)

# T2I Styles Model
#download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_style_sd14v1.pth", style_path)

# CLIPVision model (needed for styles model)
#download("https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin", clip_vision_path, "clip_vit14.bin")


# ControlNet
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_seg_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors", controlnet_path)
#download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11u_sd15_tile_fp16.safetensors", controlnet_path)

# ControlNet SDXL
#download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors", controlnet_path)
#download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors", controlnet_path)
#download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors", controlnet_path)
#download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors", controlnet_path)

# Controlnet Preprocessor nodes by Fannovel16
#!cd custom_nodes && git clone https://github.com/Fannovel16/comfy_controlnet_preprocessors; cd comfy_controlnet_preprocessors && python install.py


# GLIGEN
#download("https://huggingface.co/comfyanonymous/GLIGEN_pruned_safetensors/resolve/main/gligen_sd14_textbox_pruned_fp16.safetensors", gligen_path)


# ESRGAN upscale model
#download("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", upscale_path)
#download("https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth", upscale_path)
#download("https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth", upscale_path)


File /content/drive/MyDrive/AI/models/checkpoints/v1-5-pruned-emaonly.ckpt already exists, skipping download.
File /content/drive/MyDrive/AI/models/vae/vae-ft-mse-840000-ema-pruned.safetensors already exists, skipping download.


### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 32.0 MB/s eta 0:00:00


In [ ]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory /content/drive/MyDrive/AI/Images

--2026-02-19 03:05:25--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb [following]
--2026-02-19 03:05:25--  https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/4fddf4d7-e02d-44dc-9e5a-ef9e28afdd54?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-19T04%3A01%3A10Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

<IPython.core.display.Javascript object>

to open it in a window you can open this link here:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

The password/enpoint ip for localtunnel is: 34.7.187.207

ComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)

your url is: https://thick-berries-allow.loca.lt
FETCH ComfyRegistry Data: 5/126
This is the URL to access ComfyUI: https://michael-penalty-alternative-recommends.trycloudflare.com                          |
This is the URL to access ComfyUI: https://models-made-bedford-gage.trycloudflare.com                                        |
This is the URL to access ComfyUI: https://magnetic-airports-convenient-wheels.trycloudflare.com                             |
FETCH ComfyRegistry Data: 10/126
FETCH ComfyRegistry Data: 15/126
FETCH ComfyRegistry Data: 20/126
FETCH ComfyRegistry Data: 25/126
[DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[DE

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory /content/drive/MyDrive/AI/Images

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴Traceback (most recent call last):
  File "/content/drive/MyDrive/AI/ComfyUI/main.py", line 55, in <module>
    import cuda_malloc
  File "/content/drive/MyDrive/AI/ComfyUI/cuda_malloc.py", line 6, in <module>
    import comfy_aimdo.control
ModuleNotFoundError: No module named 'comfy_aimdo'


### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory /content/drive/MyDrive/AI/Images

Traceback (most recent call last):
  File "/content/drive/MyDrive/AI/ComfyUI/main.py", line 55, in <module>
    import cuda_malloc
  File "/content/drive/MyDrive/AI/ComfyUI/cuda_malloc.py", line 6, in <module>
    import comfy_aimdo.control
ModuleNotFoundError: No module named 'comfy_aimdo'
